<a href="https://colab.research.google.com/github/olympiasau/ds2002-fa26/blob/main/2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
price_copy = df['price'].str.replace('$', '').astype(float)                   # NEW
print('revenue before dropping:', (price_copy * df['qty']).sum())               # NEW

before = len(df)
df = df.drop_duplicates()
log('dropping', 'dropped exact duplicate rows', before - len(df))

price_copy = df['price'].str.replace('$', '').astype(float)                   # NEW
print('revenue after dropping:', (price_copy * df['qty']).sum())                # NEW

print(len(df))

revenue before dropping: 4852.5
[dropping] dropped exact duplicate rows (15 row(s))
revenue after dropping: 4594.5
300


### TODO 2 — clean `price` -> float

In [4]:
pclean = df['price'].str.startswith('$').sum()
df['price'] = df['price'].str.replace('$', '').astype(float)
log('price', "removed dollar signs and converted price to float", pclean)

print(df['price'].value_counts())

[price] removed dollar signs and converted price to float (124 row(s))
price
7.5     117
6.0      67
12.0     65
24.0     51
Name: count, dtype: int64


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
df['qty'] = pd.to_numeric(df['qty'])
print('revenue before qty step:', (df['price'] * df['qty']).sum())

before = len(df)
df = df.dropna(subset=['qty'])
log('quantity-missing', 'dropped rows with missing quantity', before - len(df))

before = len(df)
df = df[df['qty'] > 0]
log('quantity-negative', 'dropped rows with negative quantity', before - len(df))
print('revenue after quantity step:', (df['price'] * df['qty']).sum())

print(len(df))

revenue before qty step: 4594.5
[quantity-missing] dropped rows with missing quantity (12 row(s))
[quantity-negative] dropped rows with negative quantity (13 row(s))
revenue after quantity step: 4740.0
275


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())

ITEM_MAP = {
    'cheese burger': 'Cheeseburger',
    'foam finger': 'Foam Finger',
    'rain poncho': 'Rain Poncho',
}

ccitem = df['item'].isin(ITEM_MAP.keys()).sum()   # rows that will change
df['item'] = df['item'].replace(ITEM_MAP)
log('item', 'merged 6 item spellings into 3 products', ccitem)

print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] merged 6 item spellings into 3 products (126 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(df['category'].value_counts())

CAT_MAP = {
    'food': 'Food',
    'rain-gear': 'RainGear',
}
normcat = df['category'].isin(CAT_MAP.keys()).sum()
df['category'] = df['category'].replace(CAT_MAP)
log('category', "fixed spellings: food -> Food, rain-gear -> RainGear", normcat)

cat = (df['category'] == 'Apparel').sum()
df['category'] = df['category'].replace({'Apparel': 'Merch'})
log('category', 'decision: merged Apparel into Merch', cat)

print(df['category'].value_counts())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] fixed spellings: food -> Food, rain-gear -> RainGear (89 row(s))
[category] decision: merged Apparel into Merch (43 row(s))
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].nunique() == 3
assert df['category'].nunique() == 3
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']

by_cat = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
print(by_cat)
print('Total:', round(df['revenue'].sum(), 2))

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Total: 4740.0


**What I would tell the vendor:** I would reccomend stocking more food, as it brought the most revenue (1,656/4,740)

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,dropping,dropped exact duplicate rows,15
1,price,removed dollar signs and converted price to float,124
2,quantity-missing,dropped rows with missing quantity,12
3,quantity-negative,dropped rows with negative quantity,13
4,item,merged 6 item spellings into 3 products,126
5,category,"fixed spellings: food -> Food, rain-gear -> Ra...",89
6,category,decision: merged Apparel into Merch,43


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

*Dropping the duplicates changed the total revenue the most. It started at 4,852.50 and dropped to 4,594.59 (a decrease of 258)*

*I chose to merged Apparel and Merch, but a reasonable person could have kept Apparel and Merch as separate categories. While total revenue would remain the same, this choice would change the breakdown which categories are represented the strongest within the total revenue (Merch would drop from 2nd to 3rd place)* *I chose to merge them because the stand doesn't sell any clothing, so Apparel doesn't need to be a separate product line. It is more likely it could have been a synonym used for merchandise.*

